[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cioos-siooc/CPDW-VI/blob/main/docs/notebooks/workshop_validate_biodiversity_data.ipynb)

> The version on the docs site is a static render — to run the cells, open the Colab link above or clone the repo and run locally.

# 🌊 How to Validate Your Biodiversity Data
### OBIS Developer Workshop — Pyobistools

---

**What you'll learn — all 6 validation functions:**

| Block | Function | What it checks |
|-------|----------|----------------|
| 2 | `check_fields` | Required and recommended Darwin Core fields are present |
| 3 | `check_occurrence_core_and_extension` | Valid `occurrenceStatus`, `basisOfRecord`, no duplicate IDs |
| 4 | `check_eventids` | Event hierarchy is consistent (parent–child IDs) |
| 5 | `check_measurementids` | All `measurementID` values are unique |
| 6 | `check_scientificname_and_ids` | Species names and LSIDs match WoRMS |
| 7 | `check_onland` | Coordinates fall in the ocean, not on land |

**Resources:**
- 📖 [Darwin Core standard](https://dwc.tdwg.org/)
- 📋 [Darwin Core terms — OBIS manual](https://manual.obis.org/darwin_core.html)
- 🐍 [pyobistools on GitHub](https://github.com/cioos-siooc/pyobistools)
- 🌐 [WoRMS — World Register of Marine Species](https://www.marinespecies.org/)

⏱️ **Estimated time:** ~35 minutes  
📶 **Internet required for:** Block 6 (WoRMS API) and Block 7 bonus (OBIS API)  
🗺️ **Mermaid diagram below** requires internet to load from CDN

**Validation workflow** — each check feeds into the next; fix errors before moving on:

Your CSV → check_fields → check_occurrence → check_eventids → check_measurementids → check_scientificname_and_ids → check_onland → **Ready to publish**


In [1]:
# @colab-setup
# Run this cell first. On Colab it installs the deps; locally it is a no-op.
import sys
if "google.colab" in sys.modules:
    %pip install -q pyobistools nest_asyncio pyobis plotly


In [2]:
import sys

import pandas as pd
import numpy as np
import nest_asyncio
import plotly.graph_objects as go

from pyobistools.validation.check_fields import check_fields
from pyobistools.validation.check_occurrence_core_and_extension import check_occurrence_core_and_extension
from pyobistools.validation.check_eventids import check_eventids, check_extension_eventids
from pyobistools.validation.check_measurementids import check_measurementids
from pyobistools.validation.check_scientificname_and_ids import check_scientificname_and_ids
from pyobistools.validation.check_onland import check_onland

pd.set_option('max_colwidth', None)
print('All imports successful ✅')

All imports successful ✅


---
## 📂 Block 1 — Load Your Datasets
We'll use four synthetic datasets throughout this workshop — load them all now.

| Variable | File | Used in |
|----------|------|---------|
| `df_occ` | `workshop_ex_occurrence.csv` | Blocks 2, 3, 4, 6 — field checks, occurrence validation, event IDs, WoRMS |
| `df_event` | `workshop_ex_event_core.csv` | Block 4b — extension event ID check |
| `df_emof` | `workshop_ex_emof.csv` | Blocks 4b, 5 — extension event IDs, measurement IDs |
| `df_onland` | `workshop_ex_onland.csv` | Block 7 — on-land coordinate check |

In [3]:
_COLAB = "google.colab" in sys.modules
_BASE  = "https://raw.githubusercontent.com/cioos-siooc/CPDW-VI/main/docs/notebooks/"
def _csv(name):
    return (_BASE + name) if _COLAB else name

df_occ    = pd.read_csv(_csv('workshop_ex_occurrence.csv'))
df_event  = pd.read_csv(_csv('workshop_ex_event_core.csv'))
df_emof   = pd.read_csv(_csv('workshop_ex_emof.csv'))
df_onland = pd.read_csv(_csv('workshop_ex_onland.csv'))

print('Datasets loaded:')
for name, data in [('df_occ', df_occ), ('df_event', df_event), ('df_emof', df_emof), ('df_onland', df_onland)]:
    print(f'  {name:<12} {data.shape[0]:>4} rows x {data.shape[1]:>2} cols')

Datasets loaded:
  df_occ         15 rows x  9 cols
  df_event        5 rows x  5 cols
  df_emof        10 rows x  5 cols
  df_onland      25 rows x  7 cols


---
## 📝 Block 2 — `check_fields`

This function evaluates a Darwin Core DataFrame and reports:
- **Absent fields** — required or recommended columns missing entirely from the file
- **Empty values** — required or recommended columns that are present but contain blank cells

| Argument | Default | Effect |
|----------|---------|--------|
| `data` | — | DataFrame to validate |
| `analysis_type` | — | DWC file type: `'occurrence_core'`, `'event_core'`, `'occurrence_extension'`, `'extended_measurement_or_fact_extension'` |
| `level` | `'error'` | `'error'` — absent and empty **required** fields (field presence check is case-insensitive). `'warning'` — absent and empty **recommended** fields, plus any present required or recommended field with incorrect column name case. |
| `accepted_name_usage_id_check` | `False` | When `True`, suppresses the empty-`scientificNameID` error on rows where `acceptedNameUsageID` is filled instead |

**Output columns:** `field | level | row | message`
→ `row = NaN` — the column is entirely absent from the file
→ `row = N` — that specific row has an empty value in a required/recommended field

In [4]:
print(df_occ.columns)
df_occ.head()

Index(['occurrenceID', 'eventDate', 'decimalLatitude', 'decimalLongitude',
       'scientificname', 'occurrenceStatus', 'basisOfRecord',
       'scientificNameID', 'CoordinateUncertaintyInMeters'],
      dtype='object')


,occurrenceID,eventDate,decimalLatitude,decimalLongitude,scientificname,occurrenceStatus,basisOfRecord,scientificNameID,CoordinateUncertaintyInMeters
0,WS-OCC-001,2024-06-01,45.1,-64.2,Ficticia marina,present,HumanObservation,urn:lsid:marinespecies.org:taxname:99001,50
1,WS-OCC-001,2024-06-01,45.1,-64.2,Imaginarius atlanticus,present,HumanObservation,urn:lsid:marinespecies.org:taxname:99002,50
2,WS-OCC-002,2024-06-01,45.4,-63.8,Simulatus borealis,present,HumanObservation,NaN,50
3,WS-OCC-003,2024-06-02,45.6,-63.5,Exemplaria pelagica,present,HumanObservation,urn:lsid:marinespecies.org:taxname:99004,50
4,WS-OCC-004,2024-06-02,46.0,-62.9,Ficticia marina,present,HumanObservation,urn:lsid:marinespecies.org:taxname:99001,50


In [6]:
# Check for ERROR-level issues: missing required fields and empty required values
errors = check_fields(df_occ, analysis_type='occurrence_core', level='error')
print(f'Errors found: {len(errors)}')
errors

Errors found: 6


,field,level,row,message
8,countryCode,error,NaN,Required field countryCode is missing
9,kingdom,error,NaN,Required field kingdom is missing
10,geodeticDatum,error,NaN,Required field geodeticDatum is missing
0,scientificNameID,error,2,Empty value for required field scientificNameID
1,scientificNameID,error,6,Empty value for required field scientificNameID
2,scientificNameID,error,11,Empty value for required field scientificNameID


In [7]:
# Check for WARNING-level issues: missing recommended fields, incorrect column case
warnings = check_fields(df_occ, analysis_type='occurrence_core', level='warning')
print(f'Warnings found: {len(warnings)}')
warnings

Warnings found: 12


,field,level,row,message
11,minimumDepthInMeters,warning,NaN,Recommended field minimumDepthInMeters is missing
12,maximumDepthInMeters,warning,NaN,Recommended field maximumDepthInMeters is missing
14,samplingProtocol,warning,NaN,Recommended field samplingProtocol is missing
15,taxonRank,warning,NaN,Recommended field taxonRank is missing
16,organismQuantity,warning,NaN,Recommended field organismQuantity is missing
17,organismQuantityType,warning,NaN,Recommended field organismQuantityType is missing
18,datasetName,warning,NaN,Recommended field datasetName is missing
19,dataGeneralizations,warning,NaN,Recommended field dataGeneralizations is missing
20,informationWithheld,warning,NaN,Recommended field informationWithheld is missing
21,institutionCode,warning,NaN,Recommended field institutionCode is missing


---
### 🔧 Your Task — Block 2

The error output shows two types of issues in `df_occ`:
1. **Missing column** (`row = NaN`) — `geodeticDatum` is entirely absent from the file
2. **Empty values in a present required field** (`row = N`) — `scientificNameID` exists as a column but is blank on some rows

**Fix the `geodeticDatum` error** by adding that column to `df_fixed`.
Run `check_fields` again to confirm the `geodeticDatum` error disappears.

> 💡 All OBIS coordinates use **WGS84** — the value should be the string `'WGS84'`.
> The `scientificNameID` empty-value errors will remain — those require a WoRMS lookup (Block 6).

In [35]:
df_fixed = df_occ.copy()
# Add the missing geodeticDatum column


In [38]:
# 🔑 SOLUTION — Expand to reveal
df_fixed = df_occ.copy()
df_fixed['geodeticDatum'] = 'WGS84'
print('geodeticDatum column added.')
df_fixed.head(2)

geodeticDatum column added.


,occurrenceID,eventDate,decimalLatitude,decimalLongitude,scientificname,occurrenceStatus,basisOfRecord,scientificNameID,CoordinateUncertaintyInMeters,geodeticDatum
0,WS-OCC-001,2024-06-01,45.1,-64.2,Ficticia marina,present,HumanObservation,urn:lsid:marinespecies.org:taxname:99001,50,WGS84
1,WS-OCC-001,2024-06-01,45.1,-64.2,Imaginarius atlanticus,present,HumanObservation,urn:lsid:marinespecies.org:taxname:99002,50,WGS84


In [37]:
# Verify: run the check again on your fixed dataset
errors_after = check_fields(df_fixed, analysis_type='occurrence_core', level='error')
print(f'Errors before fix : {len(errors)}')
print(f'Errors after fix  : {len(errors_after)}')
print(f'\nRemaining errors:')
errors_after

Errors before fix : 6
Errors after fix  : 5

Remaining errors:


,field,level,row,message
8,countryCode,error,NaN,Required field countryCode is missing
9,kingdom,error,NaN,Required field kingdom is missing
0,scientificNameID,error,2,Empty value for required field scientificNameID
1,scientificNameID,error,6,Empty value for required field scientificNameID
2,scientificNameID,error,11,Empty value for required field scientificNameID


---
## 📋 Block 3 — `check_occurrence_core_and_extension`

This function validates controlled vocabulary and uniqueness constraints in occurrence files:

| What it checks | Valid values |
|----------------|-------------|
| Duplicate `occurrenceID` | All values must be unique |
| `occurrenceStatus` | `present` or `absent` (case-sensitive, lowercase) |
| `basisOfRecord` | `PreservedSpecimen`, `FossilSpecimen`, `LivingSpecimen`, `HumanObservation`, `MachineObservation`, `MaterialSample`, `MaterialCitation`, `MaterialEntity`, `Occurrence`, `Taxon`, `Event` (case-sensitive) |

We'll work with a synthetic dataset that contains intentional errors — just like a file a collaborator might send you.

In [9]:
# Explore the dataset — can you spot any issues?
df_occ

,occurrenceID,eventDate,decimalLatitude,decimalLongitude,scientificname,occurrenceStatus,basisOfRecord,scientificNameID,CoordinateUncertaintyInMeters
0,WS-OCC-001,2024-06-01,45.1,-64.2,Ficticia marina,present,HumanObservation,urn:lsid:marinespecies.org:taxname:99001,50
1,WS-OCC-001,2024-06-01,45.1,-64.2,Imaginarius atlanticus,present,HumanObservation,urn:lsid:marinespecies.org:taxname:99002,50
2,WS-OCC-002,2024-06-01,45.4,-63.8,Simulatus borealis,present,HumanObservation,NaN,50
3,WS-OCC-003,2024-06-02,45.6,-63.5,Exemplaria pelagica,present,HumanObservation,urn:lsid:marinespecies.org:taxname:99004,50
4,WS-OCC-004,2024-06-02,46.0,-62.9,Ficticia marina,present,HumanObservation,urn:lsid:marinespecies.org:taxname:99001,50
5,WS-OCC-005,2024-06-02,46.2,-62.5,Imaginarius atlanticus,present,FieldObservation,urn:lsid:marinespecies.org:taxname:99002,50
6,WS-OCC-006,2024-06-03,46.5,-62.0,Simulatus borealis,present,HumanObservation,NaN,50
7,WS-OCC-007,2024-06-03,46.7,-61.6,Exemplaria pelagica,present,HumanObservation,urn:lsid:marinespecies.org:taxname:99004,50
8,WS-OCC-008,2024-06-03,47.0,-61.1,Ficticia marina,presence,HumanObservation,urn:lsid:marinespecies.org:taxname:99001,50
9,WS-OCC-009,2024-06-04,47.2,-60.7,Imaginarius atlanticus,present,HumanObservation,urn:lsid:marinespecies.org:taxname:99002,50


In [10]:
# Run the validation
occ_errors = check_occurrence_core_and_extension(df_occ)
print(f'Issues found: {len(occ_errors)}')
occ_errors

Issues found: 5


,field,level,row,message
0,occurrenceid,error,0,occurrenceid WS-OCC-001 is duplicated
1,occurrenceid,error,1,occurrenceid WS-OCC-001 is duplicated
0,occurrencestatus,error,8,occurrencestatus presence is not permitted
0,basisofrecord,error,5,basisofrecord FieldObservation is not permitted
1,basisofrecord,error,11,basisofrecord Humanobservation is not permitted


---
### 🔧 Your Task — Block 3

The output above found issues across **4 different rows**:
1. A **duplicate `occurrenceID`** — two rows share the same ID
2. An **invalid `basisOfRecord`** — `FieldObservation` is not in the accepted vocabulary
3. An **invalid `occurrenceStatus`** — a value that is not `present` or `absent`
4. An **invalid `basisOfRecord` case** — `Humanobservation` has incorrect capitalisation

Fix all 4 issues in `df_occ_fixed` and run the check again to confirm 0 errors.

> 💡 Look at the `row` column in the error output — it tells you which row index to fix.  
> Use `df_occ_fixed.loc[row_number, 'column_name'] = 'new_value'` to fix individual cells.

In [51]:
df_occ_fixed = df_occ.copy()
# Fix 1: duplicate occurrenceID

# Fix 2: invalid basisOfRecord (wrong vocabulary term)

# Fix 3: invalid occurrenceStatus

# Fix 4: invalid basisOfRecord (wrong case)


In [11]:
# 🔑 SOLUTION — Expand to reveal
df_occ_fixed = df_occ.copy()
# Fix 1: make the duplicate occurrenceID unique
df_occ_fixed.loc[1, 'occurrenceID'] = 'WS-OCC-001-B'
# Fix 2: use a valid basisOfRecord vocabulary term
df_occ_fixed.loc[5, 'basisOfRecord'] = 'HumanObservation'
# Fix 3: use 'present' (lowercase) — not 'presence'
df_occ_fixed.loc[8, 'occurrenceStatus'] = 'present'
# Fix 4: correct the capitalisation
df_occ_fixed.loc[11, 'basisOfRecord'] = 'HumanObservation'


In [12]:
# Verify: should return 0 errors
result = check_occurrence_core_and_extension(df_occ_fixed)
print(f'Issues remaining: {len(result)}')
if len(result) == 0:
    print('All issues fixed! ✅')

Issues remaining: 0
All issues fixed! ✅


---
## 🔗 Block 4 — `check_eventids`

Event-based Darwin Core datasets use a **parent–child hierarchy**:

```
Cruise (top-level, no parent)
 ├── Station A  ← parentEventID = cruise eventID
 └── Station B  ← parentEventID = cruise eventID
```

Top-level events have an **empty `parentEventID`**. All other events must set
`parentEventID` to an `eventID` that actually exists in the same file.

| What it checks | Condition |
|----------------|-----------|
| `eventID` field | Must be present in the dataset |
| `parentEventID` field | Must be present in the dataset |
| `eventID` values | Must be unique — no duplicates |
| `parentEventID` values | Every non-empty value must reference an existing `eventID` |

This function is used with an **event_core** or **occurrence_core** — not an extension.
We'll use `df_event`, which has a cruise–station hierarchy with two intentional errors.

In [43]:
# Explore the event core
print(f'df_event : {df_event.shape[0]} rows x {df_event.shape[1]} cols')
print(f'Columns  : {list(df_event.columns)}')
df_event

df_event : 5 rows x 5 cols
Columns  : ['eventID', 'parentEventID', 'eventDate', 'decimalLatitude', 'decimalLongitude']


,eventID,parentEventID,eventDate,decimalLatitude,decimalLongitude
0,CRUISE-2024,NaN,2024-06-01,NaN,NaN
1,STATION-001,CRUISE-2024,2024-06-01,45.1,-64.2
2,STATION-002,CRUISE-2024,2024-06-01,45.4,-63.8
3,STATION-002,CRUISE-2024,2024-06-02,45.6,-63.5
4,STATION-003,CRUISE-9999,2024-06-02,45.8,-63.2


In [45]:
# Validate event IDs in the event core
eventid_errors = check_eventids(df_event)
print(f'Event ID issues found: {len(eventid_errors)}')
eventid_errors

Event ID issues found: 3


,field,level,row,message
2,eventid,error,2,eventid STATION-002 is duplicated
3,eventid,error,3,eventid STATION-002 is duplicated
4,parenteventid,error,4,parenteventid CRUISE-9999 has no corresponding eventID


---
### 🔧 Your Task — Block 4

The output above found **2 issues**:
1. A **duplicate `eventID`** — `STATION-002` appears on two rows
2. A **broken `parentEventID`** — `STATION-003` has `parentEventID = CRUISE-9999`,
   but no row in the file has `eventID = CRUISE-9999` (the parent event is missing)

Fix both errors in `df_event_fixed` and run `check_eventids` again to confirm 0 errors.

> 💡 For the duplicate: keep the first occurrence and drop the second.
> For the broken parent: `STATION-003` should belong to `CRUISE-2024`.

In [46]:
df_event_fixed = df_event.copy()
# Fix 1: remove the duplicate eventID

# Fix 2: correct the orphaned parentEventID


In [47]:
# 🔑 SOLUTION — Expand to reveal
df_event_fixed = df_event.copy()
# Fix 1: keep only the first occurrence of each eventID
df_event_fixed = df_event_fixed.drop_duplicates(subset='eventID', keep='first').reset_index(drop=True)
# Fix 2: STATION-003 belongs to CRUISE-2024, not the non-existent CRUISE-9999
df_event_fixed.loc[df_event_fixed['eventID'] == 'STATION-003', 'parentEventID'] = 'CRUISE-2024'
print('Fixes applied.')

eventid_errors_fixed = check_eventids(df_event_fixed)
print(f'Event ID issues remaining: {len(eventid_errors_fixed)}')
if len(eventid_errors_fixed) == 0:
    print('Event IDs are valid! ✅')
df_event_fixed


Fixes applied.
Event ID issues remaining: 0
Event IDs are valid! ✅


,eventID,parentEventID,eventDate,decimalLatitude,decimalLongitude
0,CRUISE-2024,NaN,2024-06-01,NaN,NaN
1,STATION-001,CRUISE-2024,2024-06-01,45.1,-64.2
2,STATION-002,CRUISE-2024,2024-06-01,45.4,-63.8
3,STATION-003,CRUISE-2024,2024-06-02,45.8,-63.2


---
### `check_extension_eventids`

When you publish an eMoF file alongside a core file, every `eventID` in the
extension must match an `eventID` in the core — otherwise those measurements
cannot be linked to any sampling event.

Supported file pairings:
- event_core + occurrence_extension
- event_core + eMoF
- occurrence_core + eMoF

| Argument | Default | Effect |
|----------|---------|--------|
| `core` | — | DataFrame of the event_core or occurrence_core |
| `extension_or_emof` | — | DataFrame of the eMoF or extension file |
| `field` | `'eventID'` | Linking field: `'eventID'` or `'occurrenceID'` |

`df_event` has stations `STATION-001` through `STATION-003`.
`df_emof` references `STATION-001` through `STATION-004` — let's see what happens.

In [53]:
df_event

,eventID,parentEventID,eventDate,decimalLatitude,decimalLongitude
0,CRUISE-2024,NaN,2024-06-01,NaN,NaN
1,STATION-001,CRUISE-2024,2024-06-01,45.1,-64.2
2,STATION-002,CRUISE-2024,2024-06-01,45.4,-63.8
3,STATION-002,CRUISE-2024,2024-06-02,45.6,-63.5
4,STATION-003,CRUISE-9999,2024-06-02,45.8,-63.2


In [54]:
df_emof

,eventID,measurementID,measurementType,measurementValue,measurementUnit
0,STATION-001,MEAS-001,Water temperature,12.3,Celsius
1,STATION-001,MEAS-002,Salinity,32.1,PSU
2,STATION-001,MEAS-002,Depth,5.2,metres
3,STATION-002,MEAS-003,Water temperature,11.8,Celsius
4,STATION-002,MEAS-004,Salinity,31.9,PSU
5,STATION-002,MEAS-004,Depth,3.7,metres
6,STATION-003,MEAS-006,Water temperature,13.1,Celsius
7,STATION-003,MEAS-007,Salinity,33.2,PSU
8,STATION-003,MEAS-008,Depth,8.5,metres
9,STATION-004,MEAS-009,Water temperature,10.5,Celsius


In [ ]:
# Check that all df_emof eventIDs have a match in the event core
ext_errors = check_extension_eventids(df_event, df_emof, field='eventID')
print(f'Extension linkage errors: {len(ext_errors)}')
ext_errors

Extension linkage errors: 1


,field,level,row,message
0,eventid,error,9,Field STATION-004 has no corresponding eventID in the core


---
### 🔧 Your Task — Block 4b

`df_emof` contains measurements for `STATION-004`, but that station has no
corresponding `eventID` in `df_event`.

**Fix the error** by adding `STATION-004` to `df_event_fixed` and rerunning the check.

> 💡 In a real dataset this means either:  
> - the core file is incomplete (the station was never recorded as an event), or  
> - the eMoF has a typo in one of its `eventID` values.

In [49]:
df_event_fixed = df_event.copy()
# Add the missing station to the event core


In [50]:
# 🔑 SOLUTION — Expand to reveal
df_event_fixed = pd.concat([
    df_event,
    pd.DataFrame({'eventID': ['STATION-004'], 'parentEventID': ['STATION-004']})
], ignore_index=True)

result = check_extension_eventids(df_event_fixed, df_emof, field='eventID')
print(f'Extension linkage errors remaining: {len(result)}')
if len(result) == 0:
    print('All eMoF records are linked to a core event! ✅')

Extension linkage errors remaining: 0
All eMoF records are linked to a core event! ✅


---
## 📐 Block 5 — `check_measurementids`

In the extended Measurement or Fact (eMoF) extension, each measurement record should have a **unique `measurementID`**.  
Duplicate IDs break the ability to reference individual measurements and are rejected by OBIS.

This function has one argument: `data` — the eMoF DataFrame.  
It returns a standard `[field | level | row | message]` error DataFrame.

In [4]:
# Explore the eMoF dataset — can you spot the duplicate IDs?
df_emof

,eventID,measurementID,measurementType,measurementValue,measurementUnit
0,STATION-001,MEAS-001,Water temperature,12.3,Celsius
1,STATION-001,MEAS-002,Salinity,32.1,PSU
2,STATION-001,MEAS-002,Depth,5.2,metres
3,STATION-002,MEAS-003,Water temperature,11.8,Celsius
4,STATION-002,MEAS-004,Salinity,31.9,PSU
5,STATION-002,MEAS-004,Depth,3.7,metres
6,STATION-003,MEAS-006,Water temperature,13.1,Celsius
7,STATION-003,MEAS-007,Salinity,33.2,PSU
8,STATION-003,MEAS-008,Depth,8.5,metres
9,STATION-004,MEAS-009,Water temperature,10.5,Celsius


In [5]:
# Run the check
meas_errors = check_measurementids(df_emof)
print(f'Measurement ID issues found: {len(meas_errors)}')
meas_errors

Measurement ID issues found: 4


,field,level,row,message
0,measurementid,error,1,measurementid MEAS-002 is duplicated
1,measurementid,error,2,measurementid MEAS-002 is duplicated
2,measurementid,error,4,measurementid MEAS-004 is duplicated
3,measurementid,error,5,measurementid MEAS-004 is duplicated


---
### 🔧 Your Task — Block 5

`MEAS-002` and `MEAS-004` each appear on two rows — the check flagged all 4 offending rows.

**Assign unique `measurementID` values** to all rows in `df_emof_fixed`.  
Run the check again to confirm 0 errors.

> 💡 There are many valid approaches. The simplest is to generate a new sequential ID for every row.

In [6]:
df_emof_fixed = df_emof.copy()
# Assign unique measurementIDs


In [7]:
# 🔑 SOLUTION — Expand to reveal
df_emof_fixed = df_emof.copy()
df_emof_fixed['measurementID'] = [f'MEAS-{i:03d}' for i in range(1, len(df_emof_fixed) + 1)]

In [8]:
# Verify: should return 0 errors
result = check_measurementids(df_emof_fixed)
print(f'Measurement ID issues remaining: {len(result)}')
if len(result) == 0:
    print('All measurementIDs are now unique! ✅')
df_emof_fixed

Measurement ID issues remaining: 0
All measurementIDs are now unique! ✅


,eventID,measurementID,measurementType,measurementValue,measurementUnit
0,STATION-001,MEAS-001,Water temperature,12.3,Celsius
1,STATION-001,MEAS-002,Salinity,32.1,PSU
2,STATION-001,MEAS-003,Depth,5.2,metres
3,STATION-002,MEAS-004,Water temperature,11.8,Celsius
4,STATION-002,MEAS-005,Salinity,31.9,PSU
5,STATION-002,MEAS-006,Depth,3.7,metres
6,STATION-003,MEAS-007,Water temperature,13.1,Celsius
7,STATION-003,MEAS-008,Salinity,33.2,PSU
8,STATION-003,MEAS-009,Depth,8.5,metres
9,STATION-004,MEAS-010,Water temperature,10.5,Celsius


---
## 🧬 Block 6 — `check_scientificname_and_ids`

This function queries the **World Register of Marine Species (WoRMS)** to validate:
- Whether scientific names are recognized
- Whether `scientificNameID` (LSID) matches the accepted name
- Whether taxon rank is correct

**What is an LSID?** A globally unique persistent identifier: `urn:lsid:marinespecies.org:taxname:126505`

| `value` | Returns | What it checks |
|---------|---------|----------------|
| `'names'` | 1 DataFrame | Name recognized in WoRMS? |
| `'names_ids'` | 2 DataFrames | Above + LSID matches WoRMS? |
| `'names_taxons_ids'` | 2 DataFrames | Above + taxon rank correct? |

**Output `Oui/Yes`** = match   **`Non/No`** = mismatch

> 📶 **Internet required — live WoRMS API calls.** Expect ~30–90 seconds for this dataset.

In [9]:
nest_asyncio.apply()
print('Ready to query WoRMS 🌎')

Ready to query WoRMS 🌎


In [13]:
df_occ.scientificname.unique()

array(['Ficticia marina', 'Imaginarius atlanticus', 'Simulatus borealis',
       'Exemplaria pelagica'], dtype=object)

In [14]:
# Step 1: validate scientific names only
# All 4 species in df_occ are fictional — expect Non/No for every name
names_result = check_scientificname_and_ids(df_occ, value='names')

n_total   = len(names_result)
n_match   = (names_result[('Validation', 'Exact_Match')] == 'Oui/Yes').sum()
n_nomatch = (names_result[('Validation', 'Exact_Match')] == 'Non/No').sum()
print(f'Unique names checked  : {n_total}')
print(f'  Matched  (Oui/Yes) : {n_match}')
print(f'  Mismatch (Non/No)  : {n_nomatch}')
names_result

Unique names checked  : 4
  Matched  (Oui/Yes) : 0
  Mismatch (Non/No)  : 4


Dataset Values  Validation Database values                        \
           scientificName Exact_Match         TaxonID Status Unacceptreason   
3     Exemplaria pelagica      Non/No                                         
0         Ficticia marina      Non/No                                         
1  Imaginarius atlanticus      Non/No                                         
2      Simulatus borealis      Non/No                                         

                                            
  Taxon_Rank Valid_TaxonID Valid_Name LSID  
3                                           
0                                           
1                                           
2

---
### 🔧 Your Task — Block 6, Part 1

The `names_result` table shows a `Non/No` in `Exact_Match` for species not recognized by WoRMS.

**Write code to:**
1. Filter `names_result` to show only the `Non/No` rows
2. Print just the list of unrecognized species names

> 💡 The column is a multi-index tuple: `('Validation', 'Exact_Match')`

In [15]:
# Filter to Non/No rows and print the species names


In [16]:
# 🔑 SOLUTION — Expand to reveal
non_match = names_result[names_result[('Validation', 'Exact_Match')] == 'Non/No']
print(f'Species not recognized by WoRMS: {len(non_match)}')
print()
for name in non_match[('Dataset Values', 'scientificName')].values:
    print(f'  - {name}')

Species not recognized by WoRMS: 4

  - Exemplaria pelagica
  - Ficticia marina
  - Imaginarius atlanticus
  - Simulatus borealis


> 💡 **Working with freshwater or terrestrial data?**  
> Use `itis_usage=True` to fall back to ITIS when WoRMS has no result:  
> `check_scientificname_and_ids(df, value='names', itis_usage=True)`  
> ITIS queries add several minutes for large datasets.

---
## 🗺️ Block 7 — `check_onland`

Marine data should be in the ocean. Coordinates on land are a red flag — common causes:
- Decimal separator error (`-66,12` instead of `-66.12`)
- Swapped latitude/longitude
- Wrong coordinate reference system (not WGS84)
- Georeferencing error (point snapped to wrong location)

| Argument | Options | Effect |
|----------|---------|--------|
| `offline` | `True` / `False` | `True` = Natural Earth shoreline (fast); `False` = OBIS web service (precise) |
| `buffer` | degrees | Points within this distance of shore are considered valid |
| `report` | `True` / `False` | `True` returns error format; `False` returns the flagged rows |

We'll use a synthetic 25-record dataset. A few coordinates are *obviously* wrong — dropped into distant inland cities — while several others look plausible but still fall on land. The offline check below flags **10 of the 25**.

In [19]:
# Explore the dataset — see if you can spot the suspicious coordinates
print(f'{len(df_onland)} records')
df_onland[['occurrenceID', 'scientificName', 'decimalLatitude', 'decimalLongitude']].head(10)


25 records


,occurrenceID,scientificName,decimalLatitude,decimalLongitude
0,WS-GEO-001,Ficticia marina,47.20,-61.40
1,WS-GEO-002,Imaginarius atlanticus,47.10,-62.30
2,WS-GEO-003,Simulatus borealis,46.90,-63.70
3,WS-GEO-004,Exemplaria pelagica,46.50,-64.90
4,WS-GEO-005,Ficticia marina,-23.55,-46.63
5,WS-GEO-006,Imaginarius atlanticus,45.70,-60.20
6,WS-GEO-007,Simulatus borealis,45.00,-63.60
7,WS-GEO-008,Exemplaria pelagica,45.20,-64.10
8,WS-GEO-009,Ficticia marina,44.60,-62.50
9,WS-GEO-010,Imaginarius atlanticus,43.90,-59.80


In [20]:
# Run the on-land check (offline = Natural Earth shoreline, no extra internet needed)
flagged = check_onland(df_onland, offline=True)
print(f'Records flagged as on land: {len(flagged)}')
flagged[['occurrenceID', 'scientificName', 'decimalLatitude', 'decimalLongitude', 'on_land']]

c:\Users\simon\CPDW-VI\.venv\Lib\site-packages\cartopy\io\__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


Records flagged as on land: 10


,occurrenceID,scientificName,decimalLatitude,decimalLongitude,on_land
3,WS-GEO-004,Exemplaria pelagica,46.50,-64.90,True
4,WS-GEO-005,Ficticia marina,-23.55,-46.63,True
6,WS-GEO-007,Simulatus borealis,45.00,-63.60,True
7,WS-GEO-008,Exemplaria pelagica,45.20,-64.10,True
12,WS-GEO-013,Ficticia marina,41.88,-87.63,True
13,WS-GEO-014,Imaginarius atlanticus,44.70,-65.30,True
15,WS-GEO-016,Exemplaria pelagica,46.40,-60.80,True
17,WS-GEO-018,Imaginarius atlanticus,46.10,-62.60,True
19,WS-GEO-020,Exemplaria pelagica,19.43,-99.13,True
20,WS-GEO-021,Ficticia marina,44.40,-64.70,True


In [ ]:
# Visualize: all records blue, on-land records red
# The map zooms out to fit every point — the 3 distant-city errors stretch the view;
# the other red points sit on land within the Atlantic Canada cluster.
all_lats = df_onland['decimalLatitude'].astype(float)
all_lons = df_onland['decimalLongitude'].astype(float)

centre_lat = 0.5 * (all_lats.max() + all_lats.min())
centre_lon = 0.5 * (all_lons.max() + all_lons.min())
area = (all_lats.max() - all_lats.min()) * (all_lons.max() - all_lons.min())
zoom = np.interp(x=area, xp=[0.0005, .02, .05, 30, 350, 3500], fp=[12, 9.5, 6, 4, 2, 1])

fig = go.Figure()
fig.add_trace(go.Scattermapbox(
    lat=all_lats, lon=all_lons, mode='markers',
    marker={'size': 7, 'color': 'steelblue', 'opacity': 0.7},
    name=f'All records ({len(df_onland)})'
))
fig.add_trace(go.Scattermapbox(
    lat=flagged['decimalLatitude'].astype(float),
    lon=flagged['decimalLongitude'].astype(float), mode='markers',
    marker={'size': 12, 'color': 'red', 'opacity': 0.9},
    name=f'On land ⚠️ ({len(flagged)})'
))
fig.update_layout(
    mapbox={'style': 'open-street-map', 'center': {'lon': centre_lon, 'lat': centre_lat}, 'zoom': zoom},
    showlegend=True, legend={'x': 0.01, 'y': 0.99},
    margin={'l': 0, 'r': 0, 'b': 0, 't': 30},
    title_text='Workshop dataset — on-land observations highlighted in red',
    height=480
)
fig.show()

---
### 🔧 Your Task — Block 7

`check_onland` flagged **10** of the 25 records (stored in `flagged`). On the map, three red points sit far outside the survey area — obvious errors in distant cities — while the rest fall on land *within* Atlantic Canada: plausible-looking coordinates that are still invalid for marine data.

> ⚠️ The offline Natural Earth shoreline is **coarse**, so it can flag near-shore points generously. Re-run with `offline=False` to use OBIS's precise shoreline service, which may flag fewer.

**Create `df_ocean`** containing only the valid ocean observations (remove the on-land rows).  
Then verify with `check_onland` that no flagged records remain.

> 💡 `flagged.index` contains the original row indices of the on-land records.  
> Use `.isin()` to identify which rows to exclude from `df_onland`.

In [22]:
# Create df_ocean with on-land rows removed
df_ocean = df_onland
# Your fix here


In [23]:
# 🔑 SOLUTION — Expand to reveal
df_ocean = df_onland[~df_onland.index.isin(flagged.index)]
print(f'Removed {len(flagged)} on-land record(s).')
print(f'Ocean records remaining: {len(df_ocean)}')

Removed 10 on-land record(s).
Ocean records remaining: 15


In [24]:
# Verify: check_onland should return 0 flagged rows
flagged_after = check_onland(df_ocean, offline=True)
print(f'Flagged records remaining: {len(flagged_after)}')
if len(flagged_after) == 0:
    print('All coordinates are now in the ocean! ✅')

# Show cleaned map (all blue, no red)
clean_lats = df_ocean['decimalLatitude'].astype(float)
clean_lons = df_ocean['decimalLongitude'].astype(float)
c_lat = 0.5 * (clean_lats.max() + clean_lats.min())
c_lon = 0.5 * (clean_lons.max() + clean_lons.min())
c_area = (clean_lats.max() - clean_lats.min()) * (clean_lons.max() - clean_lons.min())
c_zoom = np.interp(x=c_area, xp=[0.0005, .02, .05, 30, 350, 3500], fp=[12, 9.5, 6, 4, 2, 1])

fig2 = go.Figure(go.Scattermapbox(
    lat=clean_lats, lon=clean_lons, mode='markers',
    marker={'size': 8, 'color': 'steelblue', 'opacity': 0.8},
    name=f'Ocean records ({len(df_ocean)})'
))
fig2.update_layout(
    mapbox={'style': 'open-street-map', 'center': {'lon': c_lon, 'lat': c_lat}, 'zoom': c_zoom},
    showlegend=True, margin={'l': 0, 'r': 0, 'b': 0, 't': 30},
    title_text='Cleaned dataset — all records in the ocean ✅',
    height=400
)
fig2.show()

Flagged records remaining: 0
All coordinates are now in the ocean! ✅


C:\Users\simon\AppData\Local\Temp\ipykernel_63164\2865842416.py:15: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig2 = go.Figure(go.Scattermapbox(


---
## 📈 Block 8 — Full Validation Pipeline

Now let's build a reusable `run_validation()` function that runs all offline checks in one pass  
and returns a single consolidated error report.

In [26]:
def run_validation(df, analysis_type='occurrence_core'):
    """Run all available offline checks and return a combined error report."""
    results = []

    for level in ('error', 'warning'):
        r = check_fields(df, level=level, analysis_type=analysis_type).copy()
        if len(r):
            r['check'] = f'check_fields ({level})'
            results.append(r)

    r = check_occurrence_core_and_extension(df).copy()
    if len(r):
        r['check'] = 'check_occurrence'
        results.append(r)

    if not results:
        print('No issues found! ✅')
        return pd.DataFrame(columns=['field', 'level', 'row', 'message', 'check'])

    return pd.concat(results, ignore_index=True)


print('run_validation() ready ✅')

run_validation() ready ✅


In [27]:
# Run the full pipeline on df_occ
report = run_validation(df_occ)
print(f'Total issues: {len(report)} ({(report["level"]=="error").sum()} errors, {(report["level"]=="warning").sum()} warnings)')

# Scoreboard: group by check, level, and field
scoreboard = (
    report
    .groupby(['check', 'level', 'field'])
    .size()
    .reset_index(name='count')
    .sort_values(['level', 'count'], ascending=[True, False])
    .reset_index(drop=True)
)
print('\n🏆 Data Quality Scoreboard')
scoreboard

Total issues: 23 (11 errors, 12 warnings)

🏆 Data Quality Scoreboard


,check,level,field,count
0,check_fields (error),error,scientificNameID,3
1,check_occurrence,error,basisofrecord,2
2,check_occurrence,error,occurrenceid,2
3,check_fields (error),error,countryCode,1
4,check_fields (error),error,geodeticDatum,1
5,check_fields (error),error,kingdom,1
6,check_occurrence,error,occurrencestatus,1
7,check_fields (warning),warning,coordinateUncertaintyInMeters,1
8,check_fields (warning),warning,dataGeneralizations,1
9,check_fields (warning),warning,datasetName,1


---
### 🔧 Your Task — Block 8

Using the scoreboard above, **apply all the fixes from previous blocks** to create a `df_final`  
and run the pipeline again. How much did the error count drop?

> 💡 You already solved the `geodeticDatum` error in Block 2. Chain that fix with any others you can address.

In [28]:
df_final = df_occ.copy()
# Apply fixes from previous blocks

# Run the pipeline
report_final = run_validation(df_final)
print(f'Issues remaining: {len(report_final)}')

Issues remaining: 23


In [29]:
# 🔑 SOLUTION — Expand to reveal
df_final = df_occ.copy()
# Fix from Block 2: add the missing required column
df_final['geodeticDatum'] = 'WGS84'

report_final = run_validation(df_final)
print(f'Before : {len(report)} issues')
print(f'After  : {len(report_final)} issues')
print(f'Fixed  : {len(report) - len(report_final)}')

Before : 23 issues
After  : 22 issues
Fixed  : 1


---
## ✅ Pre-Publication Checklist

You've run all 6 validation checks. Before publishing to OBIS:

- [ ] Fix all **`error`-level** `check_fields` issues — hard requirements
- [ ] Fix **duplicate IDs** from `check_occurrence` and `check_measurementids`
- [ ] Fix **invalid vocabulary** (`occurrenceStatus`, `basisOfRecord`) from `check_occurrence`
- [ ] Resolve **orphaned parentEventIDs** from `check_eventids`
- [ ] Correct **`scientificNameID`** to use WoRMS LSIDs from `check_scientificname_and_ids`
- [ ] Investigate and remove or document **on-land points** from `check_onland`
- [ ] Address **`warning`-level** fields (depth, sampling protocol, etc.) for richer data
- [ ] Publish via **IPT** using `pyIPT` or directly on your node's OBIS IPT instance

---

**Next tools in the workflow:**
- [pyIPT](https://github.com/cioos-siooc/pyipt) — publish to OBIS programmatically
- [OBIS2CIOOS](https://github.com/cioos-siooc/obis2cioos) — transform OBIS datasets for CIOOS discovery
- [OBIS data download API](https://api.obis.org/) — fetch data in Parquet format
- [OBIS data quality manual](https://manual.obis.org/data_standards.html)

---
> 🙋 Questions? Open an issue at [github.com/cioos-siooc/pyobistools](https://github.com/cioos-siooc/pyobistools)